# Final Testing — Multisource AI Content Intelligence Platform

This notebook performs final end-to-end testing of the AI Content Intelligence Platform.

The complete pipeline includes:

- Content ingestion
- Text cleaning
- Text chunking
- Embeddings
- Vector storage
- Semantic retrieval
- RAG-based question answering
- LLM-powered responses

The goal is to verify that the complete pipeline works correctly from
content ingestion to final question answering.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import os

from dotenv import load_dotenv

from langchain_groq import ChatGroq

from src.ingestion.pdf_loader import load_pdf
from src.processing.text_processor import split_documents
from src.embeddings.embedding_service import get_embedding_model
from src.retrieval.vector_store import create_vector_store
from src.retrieval.retriever import create_retriever
from src.rag.qa_chain import answer_question

## 1. Initialize Environment and LLM

Load the API key from the environment and initialize the Groq LLM
used for RAG-based question answering.

In [ ]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY was not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key,
    temperature=0
)


## 2. Initialize Embedding Model

Initialize the sentence-transformer embedding model used to convert
document chunks and user queries into vector representations.

In [ ]:
embedding_model = get_embedding_model()


## 3. Load Test Content

Load a PDF document as the knowledge source for the final RAG test.

In [ ]:
pdf_path = r"C:\Users\maury\Downloads\Linear Regression.pdf"

documents = load_pdf(pdf_path)

print("PDF loaded successfully.")
print("Number of pages:", len(documents))

## 4. Process and Chunk Content

Clean the extracted text and split the document into smaller chunks
for embedding and semantic retrieval.

In [ ]:
chunks = split_documents(documents)

print("Document processing completed.")
print("Number of chunks:", len(chunks))

In [ ]:
print("First chunk:")
print(chunks[0].page_content[:500])

print("\nMetadata:")
print(chunks[0].metadata)

## 5. Create Vector Store

Convert the processed document chunks into embeddings and store them
in a Chroma vector store for semantic search.

In [ ]:
vector_store = create_vector_store(
    chunks,
    embedding_model
)


## 6. Create Retriever

Create a retriever that searches the vector store for the most relevant
document chunks for a given question.

In [ ]:
retriever = create_retriever(
    vector_store,
    k=3
)


## 7. Test Semantic Retrieval

Test whether the system can retrieve relevant information from the
document before passing the retrieved context to the LLM.

In [ ]:
query = "What is linear regression?"

retrieved_documents = retriever.invoke(query)

print("Number of retrieved documents:", len(retrieved_documents))

for i, document in enumerate(retrieved_documents):
    print(f"\n--- Retrieved Document {i + 1} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)

## 8. RAG-Based Question Answering

Use the retriever to obtain relevant context and the LLM to generate
an answer based only on the retrieved content.

In [ ]:
question = "What is linear regression?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

## 9. Retrieved Sources

Display the document chunks used by the RAG pipeline to generate the
answer.

In [ ]:
print("Sources used for the answer:")

for i, document in enumerate(result["sources"]):
    print(f"\n--- Source {i + 1} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)

## 10. Additional Question Test

Test the RAG pipeline with another question related to the document.

In [ ]:
question = "What is the purpose of the cost function in linear regression?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

## 11. Test Out-of-Context Question

Verify that the system does not invent an answer when the requested
information cannot be found in the provided content.

In [ ]:
question = "What is the capital city of France?"

result = answer_question(
    question,
    retriever,
    llm
)

print("Question:")
print(question)

print("\nAnswer:")
print(result["answer"])

## 12. Final End-to-End Test

The following test runs the complete pipeline:

PDF ingestion → text processing → chunking → embeddings →
vector storage → semantic retrieval → RAG → LLM answer.

In [ ]:
final_question = "Explain the basic idea of linear regression."

final_result = answer_question(
    final_question,
    retriever,
    llm
)

print("=" * 60)
print("FINAL RAG TEST")
print("=" * 60)

print("\nQuestion:")
print(final_question)

print("\nAnswer:")
print(final_result["answer"])

print("\nNumber of sources retrieved:")
print(len(final_result["sources"]))

print("\nSource types:")

for document in final_result["sources"]:
    print(document.metadata.get("type", "unknown"))

print("\n" + "=" * 60)
print("END-TO-END TEST COMPLETED")
print("=" * 60)

The final test verifies the complete retrieval-augmented generation
pipeline.

The system successfully performs:

1. PDF content ingestion
2. Text cleaning and chunking
3. Embedding generation
4. Vector storage using Chroma
5. Semantic retrieval
6. Context-based question answering
7. Source retrieval
8. Out-of-context question handling

The RAG pipeline is now ready to be extended with additional content
intelligence capabilities such as summarization, key point extraction,
topic identification, and content insights.